In [ ]:
# %% Setup
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid libomp double-init abort (numpy and torch both bundle OpenMP on macOS)
os.environ.setdefault("OMP_NUM_THREADS", "1")           # avoids an OMP pthread_mutex_init segfault during inference on macOS

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import torch
from toto2 import Toto2Model

# Toto 2.0 (Datadog) ships as the `toto2` package inside the `toto-2` distribution, pulled in by the `toto-models`
# umbrella package (pip install toto-models). Unlike Toto 1.0's toto-ts it has no exact version pins, so it installs
# normally into this env, and its from_pretrained works on huggingface_hub>=1.0. Weights are Apache-2.0, no token needed.
# Five sizes: 4m | 22m | 313m | 1B | 2.5B. 2.5B is the largest (9.1 GB of fp32 weights) - use a smaller one on CPU.
MODEL_SIZE = "2.5B"
MAX_CONTEXT = 4096   # the context it was trained on: config.json's residual_attn_ratio = sqrt(S / ln S) for S = 128 patches of 32

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = Toto2Model.from_pretrained(f"Datadog/Toto-2.0-{MODEL_SIZE}", map_location=device).to(device).eval()
PATCH_SIZE = pipeline.config.patch_size   # 32 - the context length must be a whole number of patches
print(f"Loaded Toto-2.0-{MODEL_SIZE} on {device}")

In [ ]:
# %% Load and align data: states (targets) + exogenous inputs (covariates)
STATE_COLUMNS = {"C_A": "C_A", "T": "T", "T_C": "T_C", "h": "h", "Q_vec": "Q", "Q_c_vec": "Qc"}
EXOG_COLUMNS = {"F1": "E_R", "F2": "U_Ac", "F4": "T_F", "F5": "C_F", "F6": "T_CF", "F7": "Q_F"}

DOWNSAMPLE = 20  # must match FACTOR in CSTR-Downsample.ipynb

with h5py.File(f"CSTR_SimulationData_ds{DOWNSAMPLE}.h5", "r") as f:
    data = {col: f[key][:, 0] for key, col in STATE_COLUMNS.items()}
    n = len(data["C_A"])

# state[j] corresponds to raw input index 1 + j, strided by DOWNSAMPLE to match the full-resolution input file
idx = 1 + np.arange(n) * DOWNSAMPLE

with h5py.File("CSTR_InputVectors.h5", "r") as f:
    for key, col in EXOG_COLUMNS.items():
        data[col] = f[key][idx, 0]
    flags = {key: f[f"{key}.plt"][idx] for key in EXOG_COLUMNS}

df = pd.DataFrame(data)
df["item_id"] = "cstr"
df["timestamp"] = pd.date_range("2026-01-01", periods=n, freq=f"{DOWNSAMPLE}s")
df = df[["item_id", "timestamp"] + list(STATE_COLUMNS.values()) + list(EXOG_COLUMNS.values())]
df.head()

In [ ]:
# %% Pick a forecast window
events = []
for key in EXOG_COLUMNS:
    active = np.where(flags[key] > 0)[0]
    if len(active) == 0:
        continue
    breaks = np.where(np.diff(active) > 1)[0]
    starts = np.concatenate([[active[0]], active[breaks + 1]])
    ends = np.concatenate([active[breaks], [active[-1]]])
    events += [(s, e, EXOG_COLUMNS[key]) for s, e in zip(starts, ends)]
events.sort()

EVENT_INDEX = 12     # which fault event to forecast across, in time order; -1 = last = most available context
BUFFER = 100          # samples of steady state before the fault kept inside the forecast window (not the context)
PLOT_LEAD_IN = 500    # samples of history shown in the plot before the forecast - display only, doesn't affect what the model sees
HORIZON = None        # forecast horizon in samples - set an int yourself, or leave None to size it to the event (capped below)

event_start, event_end, event_name = events[EVENT_INDEX]
SPLIT = max(event_start - BUFFER, 0)          # everything before this is context
context_available = SPLIT
context_used = min(context_available, MAX_CONTEXT) // PATCH_SIZE * PATCH_SIZE   # rounded down to whole patches
if HORIZON is None:
    HORIZON = min((event_end - SPLIT) + 1000, n - SPLIT)   # Toto 2.0 decodes block by block, so no fixed max prediction length

print(
    f"Forecasting across the {event_name} event: {context_available} samples of context available "
    f"({context_used} will actually be used - model max is {MAX_CONTEXT}), horizon = {HORIZON} steps"
)

In [ ]:
# %% Run Toto 2.0 with past states + past-and-future exogenous inputs
# Toto 2.0 is natively multivariate: the 6 states go in as the target variates and the 6 exogenous inputs as
# `known_dynamic` variates spanning context + horizon, all under one series id so they attend to each other.
# One call covers all 6 states.
# NOTE: Datadog's README says exogenous-variable support is "planned for a future 2.0 release". forecast() already
# accepts known_dynamic, but the released weights weren't trained for it, so to the model the exog channels are
# just extra variates that happen to be observed into the future. Set USE_FUTURE_EXOG = False to pass them as
# past-only variates instead (the model then forecasts them too, and those forecasts are dropped).
# The model predicts each 32-step patch from the patches before it, so a change in an input is only seen from the
# patch after it happens, like Toto 1.0 - unlike TimesFM 3.0 / t0 / TiRex-2, which see the whole future window up front.
# Context is sliced to context_used here since the model has no internal truncation.
# Toto 2.0 outputs the deciles 0.1 .. 0.9 directly (no sampling, unlike Toto 1.0); the median is the point forecast.
QUANTILES = [0.1, 0.5, 0.9]
USE_FUTURE_EXOG = True
DECODE_BLOCK_SIZE = 768   # steps decoded per forward pass (multiple of 32), median fed back between blocks - Datadog's default

context_df = df.iloc[SPLIT - context_used:SPLIT].copy()
future_df = df.iloc[SPLIT:SPLIT + HORIZON][["item_id", "timestamp"] + list(EXOG_COLUMNS.values())].copy()
truth_df = df.iloc[SPLIT:SPLIT + HORIZON].copy()

past_states = context_df[list(STATE_COLUMNS.values())].to_numpy(dtype=np.float32).T   # (6, context_used)
past_exog = context_df[list(EXOG_COLUMNS.values())].to_numpy(dtype=np.float32).T      # (6, context_used)
if USE_FUTURE_EXOG:
    # forecast() works in whole patches, so the known future is edge-padded to cover the last partial patch
    n_future = -(-HORIZON // PATCH_SIZE) * PATCH_SIZE
    future_exog = np.pad(future_df[list(EXOG_COLUMNS.values())].to_numpy(dtype=np.float32).T, ((0, 0), (0, n_future - HORIZON)), mode="edge")
    target = torch.tensor(past_states, device=device)[None]                                         # (1, 6, context_used)
    known = torch.tensor(np.concatenate([past_exog, future_exog], axis=1), device=device)[None]     # (1, 6, context_used + n_future)
    inputs = {
        "target": target,
        "target_mask": torch.ones_like(target, dtype=torch.bool),
        "series_ids": torch.zeros(1, target.shape[1], dtype=torch.long, device=device),
        "known_dynamic": known,
        "known_dynamic_mask": torch.ones_like(known, dtype=torch.bool),
        "known_dynamic_series_ids": torch.zeros(1, known.shape[1], dtype=torch.long, device=device),
    }
else:
    target = torch.tensor(np.concatenate([past_states, past_exog]), device=device)[None]           # (1, 12, context_used): states then exog
    inputs = {
        "target": target,
        "target_mask": torch.ones_like(target, dtype=torch.bool),
        "series_ids": torch.zeros(1, target.shape[1], dtype=torch.long, device=device),
    }

out = pipeline.forecast(
    inputs,
    horizon=HORIZON,
    decode_block_size=DECODE_BLOCK_SIZE,
    has_missing_values=False,   # no gaps in the context - lets it use the fast attention kernels
)   # (9, 1, n_target_variates, HORIZON) for deciles 0.1 .. 0.9 - the first 6 variates are the states


levels = pipeline.output_head.knots
quantiles = {q: out[levels.index(q), 0].float().cpu().numpy() for q in QUANTILES}   # each (n_target_variates, HORIZON)
pred_df = pd.concat(
    [
        pd.DataFrame({
            "item_id": "cstr",
            "timestamp": truth_df["timestamp"].values,
            "target_name": col,
            "predictions": quantiles[0.5][v],
            **{str(q): quantiles[q][v] for q in QUANTILES},
        })
        for v, col in enumerate(STATE_COLUMNS.values())
    ],
    ignore_index=True,
)


pred_df.head()

In [ ]:
# %% Accuracy vs. the held-out actual simulation, and plots per state
for col in STATE_COLUMNS.values():
    sub = pred_df[pred_df["target_name"] == col].reset_index(drop=True)
    mae = np.mean(np.abs(sub["predictions"].values - truth_df[col].values))
    print(f"{col}: MAE = {mae:.5g}")

context_tail = context_df.iloc[-PLOT_LEAD_IN:]
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
for ax, col in zip(axes.ravel(), STATE_COLUMNS.values()):
    sub = pred_df[pred_df["target_name"] == col]
    ax.plot(context_tail["timestamp"], context_tail[col], color="black", label="history")
    ax.plot(truth_df["timestamp"], truth_df[col], color="black", linestyle="--", label="actual (held out)")
    ax.plot(sub["timestamp"], sub["predictions"], color="tab:blue", label="Toto 2.0 forecast")
    ax.fill_between(sub["timestamp"], sub["0.1"], sub["0.9"], color="tab:blue", alpha=0.2, label="10-90% interval")
    ax.axvline(truth_df["timestamp"].iloc[0], color="gray", linestyle=":")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
axes.ravel()[0].legend(fontsize=8)
fig.tight_layout()